# Clase 220 — Recall@k, Precision@k, MAP@k, NDCG@k, coverage, diversity

Implementación de las 6 métricas + comparativa entre 3 recomendadores sintéticos.

In [ ]:
import numpy as np

def precision_at_k(rel, k):
    """rel: array binario de relevancia en orden de ranking. precision sobre top-k."""
    return rel[:k].sum() / k

def recall_at_k(rel, k, n_relevants):
    return rel[:k].sum() / max(n_relevants, 1)

def ap_at_k(rel, k):
    """Average Precision: precision en posiciones donde hay un relevante, promediada."""
    rel_k = rel[:k]
    if rel_k.sum() == 0: return 0.0
    precisions = [(rel_k[:i+1].sum() / (i+1)) * rel_k[i] for i in range(k)]
    return sum(precisions) / min(k, rel.sum())

def dcg_at_k(rel, k):
    rel_k = rel[:k]
    return float(np.sum(rel_k / np.log2(np.arange(2, k + 2))))

def ndcg_at_k(rel, k):
    ideal = np.sort(rel)[::-1]
    idcg = dcg_at_k(ideal, k)
    return dcg_at_k(rel, k) / idcg if idcg > 0 else 0.0

## 1. Ejemplos pedagógicos

In [ ]:
examples = {
    'todos arriba':   np.array([1, 1, 1, 0, 0, 0, 0, 0, 0, 0]),
    'todos abajo':    np.array([0, 0, 0, 0, 0, 0, 0, 1, 1, 1]),
    'mezclados':      np.array([1, 0, 1, 0, 1, 0, 0, 0, 0, 0]),
    'ninguno':        np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
}
k = 10
print(f'{"caso":15} {"P@10":>6} {"R@10":>6} {"MAP@10":>8} {"NDCG@10":>8}')
for name, rel in examples.items():
    n_rel = rel.sum()
    print(f'{name:15} {precision_at_k(rel, k):>6.3f} '
          f'{recall_at_k(rel, k, n_rel):>6.3f} '
          f'{ap_at_k(rel, k):>8.3f} '
          f'{ndcg_at_k(rel.astype(float), k):>8.3f}')

print('\n→ "todos arriba" y "todos abajo" tienen igual P@10 y R@10,')
print('  pero distinto MAP y NDCG — porque éstos PENALIZAN posiciones bajas.')

## 2. Comparar 3 recomendadores sintéticos

In [ ]:
rng = np.random.default_rng(42)
n_users, n_items = 200, 500

# Generar test: cada user tiene ~5 items relevantes
R_test = np.zeros((n_users, n_items))
for u in range(n_users):
    relevant = rng.choice(n_items, size=5, replace=False)
    R_test[u, relevant] = 1

# Recomendador 1: random
scores_random = rng.random((n_users, n_items))

# Recomendador 2: popularity (siempre lo mismo)
popularity = rng.random(n_items)
scores_pop = np.tile(popularity, (n_users, 1))

# Recomendador 3: "smart" — un poco mejor que random (perfila correlación con relevancia)
scores_smart = 0.5 * rng.random((n_users, n_items)) + 0.5 * R_test + rng.normal(0, 0.3, R_test.shape)

In [ ]:
def evaluate(scores, R_test, k=10):
    metrics = {'precision': [], 'recall': [], 'map': [], 'ndcg': []}
    for u in range(scores.shape[0]):
        order = np.argsort(-scores[u])[:k]
        rel = R_test[u, order]
        n_rel = R_test[u].sum()
        metrics['precision'].append(precision_at_k(rel, k))
        metrics['recall'].append(recall_at_k(rel, k, n_rel))
        metrics['map'].append(ap_at_k(rel, k))
        metrics['ndcg'].append(ndcg_at_k(rel.astype(float), k))
    return {m: np.mean(v) for m, v in metrics.items()}

import pandas as pd
rows = []
for name, s in [('random', scores_random), ('popularity', scores_pop), ('smart', scores_smart)]:
    m = evaluate(s, R_test, k=10)
    rows.append({'model': name, **m})

results = pd.DataFrame(rows).round(4)
print(results.to_string(index=False))

## 3. Coverage + diversity

In [ ]:
def catalog_coverage(scores, n_items, k=10):
    recommended = set()
    for u in range(scores.shape[0]):
        top = np.argsort(-scores[u])[:k]
        recommended.update(top.tolist())
    return len(recommended) / n_items

# Diversity: 1 - avg pairwise similarity (acá usamos similitud sintética)
from sklearn.metrics.pairwise import cosine_similarity
item_emb = rng.normal(0, 1, (n_items, 16))
item_sim = cosine_similarity(item_emb)

def intra_list_diversity(scores, item_sim, k=10):
    divs = []
    for u in range(scores.shape[0]):
        top = np.argsort(-scores[u])[:k]
        pair_sims = [item_sim[top[i], top[j]] for i in range(k) for j in range(i+1, k)]
        divs.append(1 - np.mean(pair_sims))
    return float(np.mean(divs))

print(f'{"model":12} {"coverage":>10} {"diversity":>11}')
for name, s in [('random', scores_random), ('popularity', scores_pop), ('smart', scores_smart)]:
    c = catalog_coverage(s, n_items, k=10)
    d = intra_list_diversity(s, item_sim, k=10)
    print(f'{name:12} {c:>10.4f} {d:>11.4f}')

print('\n→ Popularity: coverage muy baja (siempre los mismos), diversity también baja.')
print('  Random: coverage perfecta, diversity alta, pero NDCG malísima.')
print('  Smart: balanceado — la idea es maximizar NDCG sin colapsar coverage.')

## 4. Validar contra `recmetrics` si está instalada

In [ ]:
try:
    from sklearn.metrics import ndcg_score
    # Para una user en particular, sklearn devuelve NDCG exacto
    u = 0
    s = scores_smart[u].reshape(1, -1)
    t = R_test[u].reshape(1, -1)
    sk_ndcg = ndcg_score(t, s, k=10)

    # Nuestro NDCG sobre el mismo user
    order = np.argsort(-scores_smart[u])[:10]
    my_ndcg = ndcg_at_k(R_test[u, order].astype(float), 10)
    print(f'sklearn NDCG@10 (user {u}): {sk_ndcg:.4f}')
    print(f'nuestro NDCG@10 (user {u}): {my_ndcg:.4f}')
except ImportError: pass

## Ejercicio guiado

1. Implementá **temporal split**: si tenés timestamps, ordená por fecha; primer 80% train, último 20% test. Compará vs random split — temporal es siempre más pesimista (correcto).
2. Reportá las 6 métricas para los 4 modelos de Clases 216-219 sobre MovieLens 100K. ¿Cuál gana en cada?
3. Novelty: `novelty = mean(-log2(item_popularity))`. Recomendaciones populares: baja novelty.
4. Plot trade-off: NDCG@10 vs coverage para distintos `α` del weighted hybrid (Clase 219).
5. Bonus: si tenés acceso a métricas online (CTR), correlacioná offline NDCG vs CTR. ¿Cuán predictivo es?

## Conclusiones

- Para top-N: NUNCA reportés RMSE/accuracy. Usar recall@k, NDCG@k, MAP@k.
- NDCG es la métrica default — sensible al orden, normalizada [0,1], acepta relevancia graduada.
- Coverage + diversity + novelty son guards: un modelo con NDCG perfecto pero coverage 5% está malo.
- Offline ≈ online pero no idéntico. A/B test (Clase 204) decide en producción.